In [ ]:
from ultralytics import YOLO

# Load base model (YOLOv8 nano, lightweight)
model = YOLO("yolov8n.pt")

# Training
results = model.train(
    data="data.yaml",  # path to your yaml
    epochs=150,
    patience=20,
    imgsz=640,
    batch=16,
    name="defect_detector_run",
)

In [ ]:
from ultralytics import YOLO
import os
from sklearn.metrics import confusion_matrix, classification_report

# --- Load trained model ---
model = YOLO("runs/detect/defect_detector_run/weights/best.pt")

# --- Paths ---
test_path = "../data/dataset_yolo/images/test"
good_path = "../data/good"

# --- List of good images (without defects) ---
good_imgs = set(os.listdir(good_path))  # exact names of good files

# --- Inference ---
results = model.predict(test_path, conf=0.25, iou=0.5, save=False, verbose=False)

# --- True Classes ---
y_true = []
for r in results:
    filename = os.path.basename(r.path)
    if filename in good_imgs:
        y_true.append(0)  # image without defect
    else:
        y_true.append(1)  # image with defect

# --- Binary Predictions ---
y_pred = []
for r in results:
    has_defect = len(r.boxes) > 0  # detected at least one defect
    y_pred.append(1 if has_defect else 0)

# --- Metrics ---
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["No defect", "With defect"])

print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)


In [ ]:
from ultralytics import YOLO
import numpy as np
import os
from tqdm import tqdm
import torch

# --- Load trained model ---
model = YOLO("runs/detect/defect_detector_run/weights/best.pt")

# --- Path to Test dataset ---
test_path = "../data/dataset_yolo/images/test"

# --- Execute official YOLO Validation (mAP, P, R) ---
metrics = model.val(data="data.yaml", split="test", conf=0.25, iou=0.3)

print("\n=== MAIN METRICS (YOLO) ===")
print(f"Precision (P): {np.mean(metrics.box.p):.3f}") 
print(f"Recall (R):    {np.mean(metrics.box.r):.3f}") 
print(f"mAP@50:        {metrics.box.map50:.3f}")
print(f"mAP@50-95:     {metrics.box.map:.3f}")

# ===========================================================
# 2️⃣ Detailed average IoU evaluation (per image)
# ===========================================================
print("\nCalculating average IoU per image...")

results = model.predict(test_path, conf=0.25, iou=0.3, save=False, verbose=False)

ious = []

for r in tqdm(results, desc="Evaluating IoU"):
    if r.boxes is None or r.boxes.shape[0] == 0:
        continue

    label_path = r.path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
    if not os.path.exists(label_path):
        continue

    # Load GT; skip if file is empty
    if os.stat(label_path).st_size == 0:
        continue

    gt_boxes = np.loadtxt(label_path, ndmin=2)
    if gt_boxes.shape[1] < 5:
        continue

    gt_boxes = gt_boxes[:, 1:]  # ignore class
    pred_boxes = r.boxes.xywhn.cpu().numpy()[:, :4]

    # Conversion
    def xywhn_to_xyxy(xywh):
        x, y, w, h = xywh.T
        x1 = x - w/2
        y1 = y - h/2
        x2 = x + w/2
        y2 = y + h/2
        return np.stack([x1, y1, x2, y2], axis=1)

    gt_boxes_xyxy = xywhn_to_xyxy(gt_boxes)
    pred_boxes_xyxy = xywhn_to_xyxy(pred_boxes)

    # IoU
    for pb in pred_boxes_xyxy:
        iou_vals = []
        for gb in gt_boxes_xyxy:
            inter_x1 = max(pb[0], gb[0])
            inter_y1 = max(pb[1], gb[1])
            inter_x2 = min(pb[2], gb[2])
            inter_y2 = min(pb[3], gb[3])
            inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
            pb_area = (pb[2] - pb[0]) * (pb[3] - pb[1])
            gb_area = (gb[2] - gb[0]) * (gb[3] - gb[1])
            union = pb_area + gb_area - inter_area
            if union > 0:
                iou_vals.append(inter_area / union)
        if len(iou_vals) > 0:
            ious.append(max(iou_vals))

if len(ious) > 0:
    print(f"Valid detections (IoU>0.3): {np.sum(np.array(ious) > 0.3)} / {len(ious)}")
else:
    print("\nNo valid overlapping found (verify labels and conf thresholds).")
